# Spark

Spark is a distributed data-processing engine: plans queries, splits data into partitions, runs tasks on many machines

- Written in	Scala, running on the JVM (Java Virtual Machine)
- Spark has several front doors to the same engine:
  - SQL (Spark SQL)
  - Python (**PySpark**)
  - Scala
  - Java, R

# PySpark Basic Syntax

In [0]:
from pyspark.sql import Window, functions as F
import time # for timing

## A good practice: define the catalog and schema at the beginning

In [0]:
db = "dsc592.retail"
f_name = f"{db}.fact_sales"
f_name

## getting a table

In [0]:
f = spark.table(f"{db}.fact_sales")

In [0]:
display(f.head(3))

## getting columns

In [0]:
f.select("amount", 
         F.col("customer_id").alias("cid")).show(3)

## filter rows (like WHERE in SQL)

In [0]:
f.filter(F.col("amount") > 700).show(3)
#f.select("amount").orderBy("amount", ascending=False).show(3)

## join and group by
we want to show daily revenue, which forces us to join dates to sales

In [0]:
# step 1: define the date table as a Spark DataFrame
d = spark.table(f"{db}.dim_date")
(f.join(d, f.date_key == d.date_key)
  .groupBy("date")
  .sum("amount")
  .orderBy("date")
  .show(3))

### How to order DESC?


In [0]:
(f.join(d, f.date_key == d.date_key)
  .groupBy("date").sum("amount")
  .orderBy("date", ascending=False)
  .show(3))

### how to rename "sum(amount)"

In [0]:
# solution 1
(f.join(d, f.date_key == d.date_key)
.groupBy("date")
.sum("amount")
.orderBy("date")
.withColumnRenamed("sum(amount)", "revenue")
).show(3)

In [0]:
# solution 2: cleaner
(f.join(d, f.date_key == d.date_key)
  .groupBy("date")
  .agg(F.sum("amount").alias("revenue"))
  .orderBy("date")).show(3)

In [0]:
# Exercise: daily revenue of February 2026
# hint: filter twice after agg
(f.join(d, f.date_key == d.date_key)
  .groupBy("date")
  .agg(F.sum("amount").alias("revenue"))
  .filter(F.year("date") == 2026)
  .filter(F.month("date") == 2)
  .orderBy("date")).show(3)

In [0]:
# Exercise: daily revenue of February 2026
# filter once with &
# VERY IMPORTANT to have () around each condition
(f.join(d, f.date_key == d.date_key)
  .groupBy("date")
  .agg(F.sum("amount").alias("revenue"))
  .filter( (F.year("date") == 2026) & (F.month("date") == 2))
  .orderBy("date")).show(3)


## Create new columns: withColumn

In [0]:
sales = (f                                            
  .withColumn("size_band",                            # new column is named "size_band"
      F.when(F.col("amount") >= 500, "large")         #    if amount >= 500 -> "large"
       .when(F.col("amount") >= 100, "medium")        #    else if >= 100   -> "medium"
       .otherwise("small"))                           #    else             -> "small"
  .withColumn("is_multi", F.col("quantity") >= 3)     # another new col: true/false column
     )     
display(sales.head(3))

# Transformations vs actions

| Transformations (lazy, return a DataFrame) | Actions (run a job, return a result) |
| --- | --- |
| `select`, `filter` / `where`, `withColumn` | `count()`, `collect()`, `take(n)` |
| `join`, `groupBy(...).agg(...)` | `show()`, `display()` |
| `orderBy`, `distinct`, `union` | `write.saveAsTable(...)` |
| `withColumn(... .over(window))` | `toPandas()` |

Transformations build a **plan**. Nothing runs until an **action** asks for a result.

In [0]:
t0 = time.time()
# define daily_rev as seen above
daily_rev = (f.join(d, f.date_key == d.date_key)
  .groupBy("date")
  .agg(F.sum("amount").alias("revenue"))
  .orderBy("date"))

print(f"defined in {time.time() - t0:.2f}s")   # near-instant: no data read

t0 = time.time()
daily_rev.show(3)                        # action -> the job runs now
print(f"counted in {time.time() - t0:.2f}s")

# Window functions in PySpark

In [0]:
top3 = spark.sql("""
    WITH product_rev AS (
  SELECT p.category, p.product_name,
         SUM(f.amount) AS revenue
  FROM dsc592.retail.fact_sales f
  JOIN dsc592.retail.dim_product p
    ON f.product_id = p.product_id
  GROUP BY p.category, p.product_name
)
SELECT *,
       ROW_NUMBER() OVER (
         PARTITION BY category
         ORDER BY revenue DESC) AS rn
FROM product_rev
QUALIFY rn <= 3
""")
top3.show(3)

In [0]:
p = spark.table("dsc592.retail.dim_product")
product_rev = (f.join(p, on="product_id")
  .groupBy("category", "product_name")
  .agg(F.sum("amount").alias("revenue")))

w = (Window.partitionBy("category")
           .orderBy(F.desc("revenue")))

top3_sp = (product_rev
  .withColumn("rn", F.row_number().over(w))
  .filter(F.col("rn") <= 3))
display(top3_sp)